# Day 23 — Final Validation and Performance Comparison

## Objective

The objective of Day 23 is to evaluate the frozen final models on the held-out test data, measure runtime and model size, compare the final methods against the strongest simple baselines, and document the quality-versus-complexity trade-off.

## Evaluation Policy

- Frozen preprocessing is used without modification.
- Frozen model configurations are used without modification.
- The held-out test set is used only for final evaluation.
- No hyperparameter tuning is performed using test results.
- Results are recorded for reproducibility.

## Final Models

### Anomaly Detection
- Final: Tuned Isolation Forest
- Simple baseline: Statistical Z-score

### Forecasting
- Final: Tuned Random Forest
- Simple baseline: Persistence

## Metrics

### Anomaly Detection
- Anomaly rate
- Processing/inference time
- Model size

Ground-truth anomaly labels are unavailable, so accuracy, precision, recall and F1 cannot be calculated for anomaly detection.

### Forecasting
- MAE
- RMSE
- Inference time
- Model size

## Quality-versus-Complexity Analysis

The final methods will be compared with simple baselines using their measured predictive/anomaly behaviour, runtime and model size.

In [1]:
import os
import time
import joblib
import numpy as np
import pandas as pd

from sklearn.metrics import mean_absolute_error, mean_squared_error

project_path = r"C:\Users\MESHWA\Downloads\underwater-water-quality-ai-analytics"

models_path = project_path + r"\models"
artifacts_path = project_path + r"\artifacts\day22_final"
data_path = project_path + r"\data"

print("Libraries loaded.")

Libraries loaded.


In [2]:
final_if_path = artifacts_path + r"\final_isolation_forest.joblib"
final_rf_path = artifacts_path + r"\final_random_forest.joblib"

final_if = joblib.load(final_if_path)
final_rf = joblib.load(final_rf_path)

print("Frozen Isolation Forest loaded.")
print("Frozen Random Forest loaded.")

print("\nIsolation Forest configuration:")
print(final_if["configuration"])

print("\nRandom Forest configuration:")
print(final_rf["configuration"])

Frozen Isolation Forest loaded.
Frozen Random Forest loaded.

Isolation Forest configuration:
{'n_estimators': 100, 'contamination': 0.03, 'max_samples': 512}

Random Forest configuration:
{'n_estimators': 300, 'max_depth': 5, 'min_samples_split': 5, 'min_samples_leaf': 2}


In [3]:
print("Files in data folder:")
for root, dirs, files in os.walk(data_path):
    for file in files:
        print(os.path.join(root, file))

Files in data folder:
C:\Users\MESHWA\Downloads\underwater-water-quality-ai-analytics\data\processed\test_v1.csv
C:\Users\MESHWA\Downloads\underwater-water-quality-ai-analytics\data\processed\test_v1_engineered.csv
C:\Users\MESHWA\Downloads\underwater-water-quality-ai-analytics\data\processed\train_v1.csv
C:\Users\MESHWA\Downloads\underwater-water-quality-ai-analytics\data\processed\train_v1_engineered.csv
C:\Users\MESHWA\Downloads\underwater-water-quality-ai-analytics\data\processed\validation_v1.csv
C:\Users\MESHWA\Downloads\underwater-water-quality-ai-analytics\data\processed\validation_v1_engineered.csv
C:\Users\MESHWA\Downloads\underwater-water-quality-ai-analytics\data\processed\water_quality_timestamped.csv
C:\Users\MESHWA\Downloads\underwater-water-quality-ai-analytics\data\raw\water_potability.csv


In [4]:
for root, dirs, files in os.walk(project_path):
    for file in files:
        if "test" in file.lower():
            print(os.path.join(root, file))

C:\Users\MESHWA\Downloads\underwater-water-quality-ai-analytics\.venv\Lib\site-packages\colorama\tests\ansitowin32_test.py
C:\Users\MESHWA\Downloads\underwater-water-quality-ai-analytics\.venv\Lib\site-packages\colorama\tests\ansi_test.py
C:\Users\MESHWA\Downloads\underwater-water-quality-ai-analytics\.venv\Lib\site-packages\colorama\tests\initialise_test.py
C:\Users\MESHWA\Downloads\underwater-water-quality-ai-analytics\.venv\Lib\site-packages\colorama\tests\isatty_test.py
C:\Users\MESHWA\Downloads\underwater-water-quality-ai-analytics\.venv\Lib\site-packages\colorama\tests\winterm_test.py
C:\Users\MESHWA\Downloads\underwater-water-quality-ai-analytics\.venv\Lib\site-packages\colorama\tests\__pycache__\ansitowin32_test.cpython-311.pyc
C:\Users\MESHWA\Downloads\underwater-water-quality-ai-analytics\.venv\Lib\site-packages\colorama\tests\__pycache__\ansi_test.cpython-311.pyc
C:\Users\MESHWA\Downloads\underwater-water-quality-ai-analytics\.venv\Lib\site-packages\colorama\tests\__pycache_

In [5]:
for root, dirs, files in os.walk(project_path):
    for file in files:
        if "test" in file.lower():
            print(os.path.join(root, file))

C:\Users\MESHWA\Downloads\underwater-water-quality-ai-analytics\.venv\Lib\site-packages\colorama\tests\ansitowin32_test.py
C:\Users\MESHWA\Downloads\underwater-water-quality-ai-analytics\.venv\Lib\site-packages\colorama\tests\ansi_test.py
C:\Users\MESHWA\Downloads\underwater-water-quality-ai-analytics\.venv\Lib\site-packages\colorama\tests\initialise_test.py
C:\Users\MESHWA\Downloads\underwater-water-quality-ai-analytics\.venv\Lib\site-packages\colorama\tests\isatty_test.py
C:\Users\MESHWA\Downloads\underwater-water-quality-ai-analytics\.venv\Lib\site-packages\colorama\tests\winterm_test.py
C:\Users\MESHWA\Downloads\underwater-water-quality-ai-analytics\.venv\Lib\site-packages\colorama\tests\__pycache__\ansitowin32_test.cpython-311.pyc
C:\Users\MESHWA\Downloads\underwater-water-quality-ai-analytics\.venv\Lib\site-packages\colorama\tests\__pycache__\ansi_test.cpython-311.pyc
C:\Users\MESHWA\Downloads\underwater-water-quality-ai-analytics\.venv\Lib\site-packages\colorama\tests\__pycache_

In [6]:
test_path = project_path + r"\data\processed\test_v1_engineered.csv"

test_df = pd.read_csv(test_path)

print("Test dataset shape:", test_df.shape)
print("\nColumns:")
print(test_df.columns.tolist())

test_df.head()

Test dataset shape: (491, 42)

Columns:
['timestamp', 'ph', 'Hardness', 'Solids', 'Chloramines', 'Sulfate', 'Conductivity', 'Organic_carbon', 'Trihalomethanes', 'Turbidity', 'Potability', 'hour', 'day_of_week', 'day', 'month', 'ph_lag_1', 'Hardness_lag_1', 'Solids_lag_1', 'Chloramines_lag_1', 'Sulfate_lag_1', 'Conductivity_lag_1', 'Organic_carbon_lag_1', 'Trihalomethanes_lag_1', 'Turbidity_lag_1', 'ph_rolling_mean_3', 'Hardness_rolling_mean_3', 'Solids_rolling_mean_3', 'Chloramines_rolling_mean_3', 'Sulfate_rolling_mean_3', 'Conductivity_rolling_mean_3', 'Organic_carbon_rolling_mean_3', 'Trihalomethanes_rolling_mean_3', 'Turbidity_rolling_mean_3', 'ph_rolling_std_3', 'Hardness_rolling_std_3', 'Solids_rolling_std_3', 'Chloramines_rolling_std_3', 'Sulfate_rolling_std_3', 'Conductivity_rolling_std_3', 'Organic_carbon_rolling_std_3', 'Trihalomethanes_rolling_std_3', 'Turbidity_rolling_std_3']


,timestamp,ph,Hardness,Solids,Chloramines,Sulfate,Conductivity,Organic_carbon,Trihalomethanes,Turbidity,...,Turbidity_rolling_mean_3,ph_rolling_std_3,Hardness_rolling_std_3,Solids_rolling_std_3,Chloramines_rolling_std_3,Sulfate_rolling_std_3,Conductivity_rolling_std_3,Organic_carbon_rolling_std_3,Trihalomethanes_rolling_std_3,Turbidity_rolling_std_3
0,2025-04-27 15:45:00,1.012023,-0.315616,-0.958172,-0.514202,-1.856349,-0.417902,-0.561794,-0.831607,-1.131728,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025-04-27 16:45:00,1.780720,1.797019,-1.095505,-0.700635,-0.986429,1.864223,-0.778065,0.853534,-1.470263,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2025-04-27 17:45:00,-1.132751,-0.349812,1.603624,1.598660,0.248355,-0.979807,-0.077711,0.853534,-0.416915,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2025-04-27 18:45:00,-0.982913,0.346949,1.700777,0.329652,1.693235,1.060005,-0.224184,1.137825,-0.069475,...,-1.006302,1.509926,1.229721,1.520250,1.277086,1.057610,1.506228,0.358609,0.972917,0.537759
4,2025-04-27 19:45:00,0.987973,0.994535,1.258097,0.669762,1.693235,-1.216393,0.857541,1.137825,-0.324616,...,-0.652217,1.640551,1.095222,1.587132,1.151711,1.341204,1.466067,0.369399,0.164135,0.729436


In [9]:
forecast_features = [
    "ph",
    "Hardness",
    "Solids",
    "Chloramines",
    "Sulfate",
    "Conductivity",
    "Organic_carbon",
    "Trihalomethanes",
    "Turbidity",
    "hour",
    "day_of_week",
    "day",
    "month",
    "ph_lag_1",
    "ph_rolling_mean_3",
    "ph_rolling_std_3"
]

print("Forecast features loaded:", len(forecast_features))
print(forecast_features)

Forecast features loaded: 16
['ph', 'Hardness', 'Solids', 'Chloramines', 'Sulfate', 'Conductivity', 'Organic_carbon', 'Trihalomethanes', 'Turbidity', 'hour', 'day_of_week', 'day', 'month', 'ph_lag_1', 'ph_rolling_mean_3', 'ph_rolling_std_3']


In [10]:
forecast_test = test_df.copy()

forecast_test["ph_next"] = forecast_test["ph"].shift(-1)

forecast_test = forecast_test.dropna(
    subset=forecast_features + ["ph_next"]
).reset_index(drop=True)

X_test_final = forecast_test[forecast_features]
y_test_final = forecast_test["ph_next"]

print("Forecast test shape:", X_test_final.shape)
print("Target shape:", y_test_final.shape)

Forecast test shape: (487, 16)
Target shape: (487,)


In [11]:
start_time = time.perf_counter()

rf_predictions = final_rf["model"].predict(
    final_rf["imputer"].transform(X_test_final)
)

rf_inference_time = time.perf_counter() - start_time

rf_mae = mean_absolute_error(
    y_test_final,
    rf_predictions
)

rf_rmse = np.sqrt(
    mean_squared_error(
        y_test_final,
        rf_predictions
    )
)

print("FINAL RANDOM FOREST")
print("=" * 40)
print("MAE:", round(rf_mae, 6))
print("RMSE:", round(rf_rmse, 6))
print("Inference time (sec):", round(rf_inference_time, 6))
print("Samples:", len(X_test_final))
print(
    "Inference time per sample (ms):",
    round((rf_inference_time / len(X_test_final)) * 1000, 6)
)

FINAL RANDOM FOREST
MAE: 0.969363
RMSE: 1.237893
Inference time (sec): 0.440977
Samples: 487
Inference time per sample (ms): 0.905498


In [13]:
start_time = time.perf_counter()

persistence_predictions = forecast_test["ph"].values

persistence_inference_time = time.perf_counter() - start_time

persistence_mae = mean_absolute_error(
    y_test_final,
    persistence_predictions
)

persistence_rmse = np.sqrt(
    mean_squared_error(
        y_test_final,
        persistence_predictions
    )
)

print("PERSISTENCE BASELINE")
print("=" * 40)
print("MAE:", round(persistence_mae, 6))
print("RMSE:", round(persistence_rmse, 6))
print("Inference time (sec):", round(persistence_inference_time, 6))

PERSISTENCE BASELINE
MAE: 1.132652
RMSE: 1.608671
Inference time (sec): 0.0003


In [15]:
forecast_comparison = pd.DataFrame({
    "Method": [
        "Persistence Baseline",
        "Tuned Random Forest"
    ],
    "MAE": [
        persistence_mae,
        rf_mae
    ],
    "RMSE": [
        persistence_rmse,
        rf_rmse
    ],
    "Inference Time (sec)": [
        persistence_inference_time,
        rf_inference_time
    ]
})

forecast_comparison

,Method,MAE,RMSE,Inference Time (sec)
0,Persistence Baseline,1.132652,1.608671,0.000300
1,Tuned Random Forest,0.969363,1.237893,0.440977


In [18]:
sensor_cols = [
    "ph",
    "Hardness",
    "Solids",
    "Chloramines",
    "Sulfate",
    "Conductivity",
    "Organic_carbon",
    "Trihalomethanes",
    "Turbidity"
]

X_test_anomaly = test_df[sensor_cols]

print("Anomaly test shape:", X_test_anomaly.shape)

Anomaly test shape: (491, 9)


In [20]:
start_time = time.perf_counter()

X_test_anomaly_imp = final_if["imputer"].transform(
    X_test_anomaly
)

if_predictions = final_if["model"].predict(
    X_test_anomaly_imp
)

if_scores = final_if["model"].decision_function(
    X_test_anomaly_imp
)

if_inference_time = time.perf_counter() - start_time

if_anomaly_count = np.sum(if_predictions == -1)
if_anomaly_rate = (if_anomaly_count / len(if_predictions)) * 100

print("FINAL ISOLATION FOREST")
print("=" * 40)
print("Test samples:", len(if_predictions))
print("Anomalies:", if_anomaly_count)
print("Anomaly rate (%):", round(if_anomaly_rate, 2))
print("Inference time (sec):", round(if_inference_time, 6))
print(
    "Inference time per sample (ms):",
    round((if_inference_time / len(if_predictions)) * 1000, 6)
)

FINAL ISOLATION FOREST
Test samples: 491
Anomalies: 12
Anomaly rate (%): 2.44
Inference time (sec): 0.029291
Inference time per sample (ms): 0.059655


In [23]:
from scipy.stats import zscore

start_time = time.perf_counter()

z_scores = np.abs(
    zscore(X_test_anomaly, nan_policy="omit")
)

z_anomaly_flags = (z_scores > 3).any(axis=1)

z_inference_time = time.perf_counter() - start_time

z_anomaly_count = np.sum(z_anomaly_flags)
z_anomaly_rate = (
    z_anomaly_count / len(z_anomaly_flags)
) * 100

print("Z-SCORE BASELINE")
print("=" * 40)
print("Test samples:", len(z_anomaly_flags))
print("Anomalies:", z_anomaly_count)
print("Anomaly rate (%):", round(z_anomaly_rate, 2))
print("Processing time (sec):", round(z_inference_time, 6))

Z-SCORE BASELINE
Test samples: 491
Anomalies: 24
Anomaly rate (%): 4.89
Processing time (sec): 0.002404


In [25]:
final_if_path = artifacts_path + r"\final_isolation_forest.joblib"
final_rf_path = artifacts_path + r"\final_random_forest.joblib"

if_model_size_mb = os.path.getsize(
    final_if_path
) / (1024 ** 2)

rf_model_size_mb = os.path.getsize(
    final_rf_path
) / (1024 ** 2)

print("Isolation Forest size (MB):", round(if_model_size_mb, 4))
print("Random Forest size (MB):", round(rf_model_size_mb, 4))

Isolation Forest size (MB): 1.6649
Random Forest size (MB): 1.1033


In [26]:
anomaly_comparison = pd.DataFrame({
    "Method": [
        "Z-score Statistical Baseline",
        "Tuned Isolation Forest"
    ],
    "Anomaly Rate (%)": [
        z_anomaly_rate,
        if_anomaly_rate
    ],
    "Processing Time (sec)": [
        z_inference_time,
        if_inference_time
    ],
    "Model Size (MB)": [
        np.nan,
        if_model_size_mb
    ]
})

anomaly_comparison

,Method,Anomaly Rate (%),Processing Time (sec),Model Size (MB)
0,Z-score Statistical Baseline,4.887984,0.002404,NaN
1,Tuned Isolation Forest,2.443992,0.029291,1.664892


In [29]:
model_size_comparison = pd.DataFrame({
    "Model": [
        "Final Isolation Forest",
        "Final Random Forest"
    ],
    "Model Size (MB)": [
        if_model_size_mb,
        rf_model_size_mb
    ]
})

model_size_comparison

,Model,Model Size (MB)
0,Final Isolation Forest,1.664892
1,Final Random Forest,1.103348


In [31]:
results_path = project_path + r"\experiments\day23_final_validation_results"

os.makedirs(results_path, exist_ok=True)

forecast_comparison.to_csv(
    results_path + r"\forecast_performance.csv",
    index=False
)

anomaly_comparison.to_csv(
    results_path + r"\anomaly_performance.csv",
    index=False
)

model_size_comparison.to_csv(
    results_path + r"\model_size_comparison.csv",
    index=False
)

print("Day 23 performance results saved successfully.")
print(results_path)

Day 23 performance results saved successfully.
C:\Users\MESHWA\Downloads\underwater-water-quality-ai-analytics\experiments\day23_final_validation_results


# Quality vs Complexity Trade-off

## Forecasting

The final Random Forest is evaluated against the Persistence baseline on the held-out test set using MAE and RMSE. The comparison is interpreted together with the measured inference time and saved model size.

The Persistence method is a simple baseline with no trained model, while the Random Forest uses a learned nonlinear model and therefore has higher model complexity and a measurable model footprint.

## Anomaly Detection

The final Isolation Forest is compared with the Z-score statistical baseline using anomaly rate and processing time.

Because ground-truth anomaly labels are unavailable, anomaly rate is not interpreted as accuracy or detection performance.

The Isolation Forest has greater model complexity and requires a saved trained model, while the Z-score method is a simpler statistical approach without a trained model.

## Final Interpretation

The final models were selected and frozen before the held-out test evaluation. The test results are reported as final validation evidence and were not used to change the frozen configurations.

The comparison therefore focuses on measured predictive quality, runtime, and model complexity rather than selecting a new model after test evaluation.

# Day 23 Final Validation Summary

## Forecasting

- Held-out test data: `test_v1_engineered.csv`
- Target: next-step pH (`ph_next`)
- Final method: Tuned Random Forest
- Baseline: Persistence
- Metrics: MAE and RMSE
- Runtime: measured on held-out test inference
- Model size: measured from the saved final artifact

## Anomaly Detection

- Held-out test data: `test_v1_engineered.csv`
- Final method: Tuned Isolation Forest
- Baseline: Z-score statistical method
- Metrics: anomaly rate and inference/processing time
- Model size: measured from the saved final artifact
- Anomaly accuracy metrics were not calculated because ground-truth anomaly labels are unavailable.

## Data and Methodology Limitations

- Timestamps were synthetically generated because the original dataset did not contain temporal information.
- The available dataset does not contain temperature, dissolved oxygen, or pressure measurements.
- Ground-truth anomaly labels are unavailable.
- The current forecasting task predicts next-step pH.
- Results should therefore be interpreted as a software/ML prototype evaluation rather than validation on real underwater sensor data.